# RAG on SEC Filings — Full-Pipeline Practice

**Goal:** build a working Retrieval-Augmented Generation pipeline end to end on real
SEC 10-K filings, and come out able to *discuss every stage* of an LLM application.

This is **not** a modeling exercise — the model just has to work well enough. The point
is fluency across the whole pipeline:

```
01 Ingestion  →  02 Embedding  →  03 Vector store  →  04 Retrieval  →  05 LLM
 docs,            encode chunks     index,              top-k,           prompt +
 chunking,        → vectors         metadata filters    rerank           context
 cleaning
```

### How this notebook teaches
Each stage has three beats:
1. **Why** — the problem the stage solves and the decision it forces.
2. **The core code is a blank** — you *write* the thing the stage teaches (marked
   `### YOUR CODE HERE`), not just read it.
3. **Instrument it** — you *measure* what your choice did (chunk counts, similarity,
   retrieved context), then a **🎤 Talk-track** note frames how you'd say it in an interview.

> **Runtime:** Google Colab with a **T4 GPU** (Runtime → Change runtime type → T4 GPU).
> **Gated model:** Stage 05 uses `mistralai/Mistral-7B-Instruct-v0.3`. Accept its license on
> Hugging Face and add your token to Colab **Secrets** as `HF_TOKEN` (key icon, left panel).
>
> Run top-to-bottom **once** — re-running the model-load cell stacks copies and can OOM the T4.

*Phase 2 (a separate exercise) adds the evaluation / experimentation layer — a gold Q/A set,
retrieval metrics, and an LLM-as-judge for faithfulness. Stubbed at the very bottom.*

## Setup

Same stack as your LangChain RAG notebook, plus `beautifulsoup4`/`lxml` (clean SEC HTML)
and `sentence-transformers`' `CrossEncoder` (the Stage-04 reranker).

In [ ]:
!pip install -qU langchain langchain-community langchain-huggingface langchain-qdrant
!pip install -qU qdrant-client sentence-transformers
!pip install -qU transformers accelerate bitsandbytes
!pip install -qU beautifulsoup4 lxml

In [2]:
import os, re, time, json, textwrap
import requests
import numpy as np
from bs4 import BeautifulSoup

from google.colab import userdata
# Add your Hugging Face token to Colab Secrets as 'HF_TOKEN' (key icon, left panel),
# or replace with: os.environ["HF_TOKEN"] = "hf_..."
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_qdrant import QdrantVectorStore
from qdrant_client.http import models as qmodels
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

import lxml

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

# SEC requires a descriptive User-Agent with a real contact. Edit to your name/email.
SEC_USER_AGENT = "David Schaaf davidschaaf@berkeley.edu"
print("Setup ready.")

Setup ready.


---
## 01 · Ingestion — docs, chunking, cleaning

**Why.** An LLM can only reason over text you put in front of it. Ingestion turns messy
source documents into clean, right-sized **chunks** with **metadata**. Two decisions live
here and both show up downstream: *how you clean* (garbage in the chunk = garbage in the
context) and *how you chunk* (too big → diluted retrieval & wasted context window; too small
→ severed meaning).

**Provided:** a helper that downloads recent 10-K filings from EDGAR (this isn't a scraping
exercise). **You write:** the cleaning and the chunking.

In [3]:
# --- PROVIDED: fetch a few 10-K filings from SEC EDGAR ---------------------
# CIKs (zero-pad to 10 digits for the submissions API).
COMPANIES = {"AAPL": 320193, "MSFT": 789019, "NVDA": 1045810}
FORM_TYPE = "10-K"

def fetch_latest_filing_html(ticker, cik, form_type=FORM_TYPE):
    headers = {"User-Agent": SEC_USER_AGENT}
    subs = requests.get(
        f"https://data.sec.gov/submissions/CIK{cik:010d}.json", headers=headers
    ).json()
    recent = subs["filings"]["recent"]
    # find the most recent filing of the requested form type
    idx = next(i for i, f in enumerate(recent["form"]) if f == form_type)
    accession = recent["accessionNumber"][idx].replace("-", "")
    primary = recent["primaryDocument"][idx]
    filing_date = recent["filingDate"][idx]
    url = f"https://www.sec.gov/Archives/edgar/data/{cik}/{accession}/{primary}"
    html = requests.get(url, headers=headers).text
    time.sleep(0.5)  # be polite to SEC (fair-access limit is 10 req/s)
    return {"ticker": ticker, "form_type": form_type,
            "filing_date": filing_date, "source_url": url, "html": html}

raw_filings = []
for tk, cik in COMPANIES.items():
    print(f"Fetching {tk} {FORM_TYPE} ...")
    raw_filings.append(fetch_latest_filing_html(tk, cik))
print(f"\nFetched {len(raw_filings)} filings.")
print("Raw HTML length (chars):", {f['ticker']: len(f['html']) for f in raw_filings})

Fetching AAPL 10-K ...
Fetching MSFT 10-K ...
Fetching NVDA 10-K ...

Fetched 3 filings.
Raw HTML length (chars): {'AAPL': 1520208, 'MSFT': 8158067, 'NVDA': 1967816}


In [4]:
# --- YOU WRITE: clean raw filing HTML into plain text ---------------------
# Turn the raw 10-K HTML into readable plain text.
#   - parse with BeautifulSoup(html, "lxml")
#   - remove <script>/<style> tags so their contents don't leak in
#   - get_text(separator=" ") and collapse whitespace with a regex
def clean_filing_text(html: str) -> str:
    ### YOUR CODE HERE
    soup = BeautifulSoup(html, features="xml")
    for tag in soup(["script", "style"]):
      tag.decompose()
    clean_soup = soup.get_text(separator=" ", strip=True)
    return re.sub(r"\s+", " ", clean_soup)


# quick sanity check on the first filing
_clean = clean_filing_text(raw_filings[0]["html"])
print("Cleaned length (chars):", len(_clean))
print(_clean[:1000])


Cleaned length (chars): 220566
aapl-20250927 false 2025 FY 0000320193 P1Y P1Y P1Y P1Y http://fasb.org/us-gaap/2025#LongTermDebtNoncurrent http://fasb.org/us-gaap/2025#LongTermDebtNoncurrent http://fasb.org/us-gaap/2025#OtherAssetsNoncurrent http://fasb.org/us-gaap/2025#OtherAssetsNoncurrent http://fasb.org/us-gaap/2025#PropertyPlantAndEquipmentNet http://fasb.org/us-gaap/2025#PropertyPlantAndEquipmentNet http://fasb.org/us-gaap/2025#OtherLiabilitiesCurrent http://fasb.org/us-gaap/2025#OtherLiabilitiesCurrent http://fasb.org/us-gaap/2025#OtherLiabilitiesNoncurrent http://fasb.org/us-gaap/2025#OtherLiabilitiesNoncurrent http://fasb.org/us-gaap/2025#OtherLiabilitiesCurrent http://fasb.org/us-gaap/2025#OtherLiabilitiesCurrent http://fasb.org/us-gaap/2025#OtherLiabilitiesNoncurrent http://fasb.org/us-gaap/2025#OtherLiabilitiesNoncurrent iso4217:USD xbrli:shares iso4217:USD xbrli:shares xbrli:pure aapl:Customer aapl:Vendor aapl:Subsidiary 0000320193 2024-09-29 2025-09-27 0000320193 us-gaap:C

In [5]:
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150

### YOUR CODE HERE
documents = []
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

for filing in raw_filings:
    metadata_base = filing.copy()
    html = metadata_base.pop('html', None) # Remove the raw HTML content from metadat

    clean_filing = clean_filing_text(html)
    chunks = text_splitter.split_text(clean_filing)

    for chunk_text in chunks:
        documents.append(Document(page_content=chunk_text, metadata=metadata_base))

print("Total chunks:", len(documents))
print(documents[99])

Total chunks: 1093
page_content='costs, limiting the Company’s ability to offer a product, service or feature to customers, imposing changes to the design of the Company’s products and services, impacting customer demand for the Company’s products and services, and requiring changes to the Company’s business or supply chain. New and changing laws, regulations, executive orders, directives, and enforcement priorities can also create uncertainty about how such laws and regulations will be interpreted and applied. If the Company is found to have violated such laws and regulations, it could materially adversely affect the Company’s business, reputation, results of operations, financial condition and stock price. Risks and costs related to new and changing laws, regulations, executive orders, directives, and enforcement priorities increase as the Company’s products and services are introduced into specialized applications, including health and financial services, or as the Company expands t

In [6]:
# --- INSTRUMENT: what did your chunking produce? -------------------------
lengths = [len(d.page_content) for d in documents]
print(f"Chunks: {len(documents)}")
print(f"Chunk length chars — min {min(lengths)}, mean {int(np.mean(lengths))}, max {max(lengths)}")
counts = {}
for d in documents:
    counts[d.metadata['ticker']] = counts.get(d.metadata['ticker'], 0) + 1
print("Chunks per company:", counts)
print("\nExample chunk + its metadata:")
print(documents[len(documents)//2].metadata)
print(textwrap.fill(documents[len(documents)//2].page_content[:500], 100))

Chunks: 1093
Chunk length chars — min 420, mean 993, max 999
Chunks per company: {'AAPL': 260, 'MSFT': 409, 'NVDA': 424}

Example chunk + its metadata:
{'ticker': 'MSFT', 'form_type': '10-K', 'filing_date': '2025-07-30', 'source_url': 'https://www.sec.gov/Archives/edgar/data/789019/000095017025100235/msft-20250630.htm'}
incurred to support and maintain cloud-based and other online products and services, including
datacenter costs and royalties; manufacturing and distribution costs for products sold and programs
licensed; operating costs related to product support service centers and product distribution
centers; traffic acquisition costs to drive traffic to our websites and to acquire online
advertising space; and costs associated with the delivery of consulting services. Research and
Development Research and d


**🎤 Talk-track — Ingestion.** *"Retrieval quality is capped by chunking. I clean the
HTML first so boilerplate and markup don't pollute the vector, then use a recursive splitter
around ~1k chars with overlap so a fact split across a boundary still survives in one chunk.
Every chunk carries metadata — ticker, form type, date — which becomes my filter and my
citation later."*

✍️ **Predict before you change it:** if you halved `CHUNK_SIZE`, what happens to the number
of chunks, to retrieval precision, and to how much context the LLM has to read?

---
## 02 · Embedding — encode chunks → vectors

**Why.** Retrieval is "find text that *means* the same thing," not "find matching keywords."
An embedding model maps each chunk to a vector so that semantically similar text lands nearby.
The model choice sets the ceiling on retrieval quality — and the vectors are what the store
in Stage 03 indexes.

In [ ]:
# --- YOU WRITE: load a sentence-embedding model --------------------------
# Use LangChain's HuggingFaceEmbeddings with "all-mpnet-base-v2"
# (a solid general-purpose sentence-transformer; 768-dim).
### YOUR CODE HERE
embeddings = HuggingFaceEmbeddings(model_name='all-mpnet-base-v2')

# sanity check: dimension + that "similar" scores higher than "unrelated"
def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

v = embeddings.embed_query("Apple reported strong iPhone revenue growth.")
print("Embedding dimension:", len(v))

In [8]:
# --- INSTRUMENT: does the embedding space behave? ------------------------
a = embeddings.embed_query("The company's revenue increased this year.")
b = embeddings.embed_query("Sales grew compared to the prior period.")   # paraphrase
c = embeddings.embed_query("The lawsuit was dismissed by the court.")     # unrelated
print(f"similar   (revenue vs sales):   {cosine(a, b):.3f}")
print(f"unrelated (revenue vs lawsuit): {cosine(a, c):.3f}")
assert cosine(a, b) > cosine(a, c), "paraphrase should score higher than the unrelated pair"
print("Embedding space looks sane ✓")

similar   (revenue vs sales):   0.740
unrelated (revenue vs lawsuit): 0.069
Embedding space looks sane ✓


**🎤 Talk-track — Embedding.** *"I use a sentence-transformer — mpnet here — because it's
tuned for semantic similarity, runs locally so there's no per-query API cost, and 768 dims is a
fine speed/quality trade. The one rule that bites people: you must embed the query with the
**same** model you embedded the corpus with, or the geometry doesn't line up."*

---
## 03 · Vector store — index, metadata filters

**Why.** You could brute-force cosine similarity over a numpy array, but a vector store gives
you an **index** for fast nearest-neighbour search at scale *and* **metadata filtering** — the
ability to say "only search Apple's filing." In a regulated setting that filter is also how you
enforce scope and access control. We use Qdrant in-memory (same as your other notebook).

In [9]:
# --- YOU WRITE: index the chunks in an in-memory Qdrant store ------------
# QdrantVectorStore.from_documents(documents, embeddings, location=":memory:",
#                                  collection_name="sec_filings")
### YOUR CODE HERE
qdrant = QdrantVectorStore.from_documents(documents, embeddings, location=":memory:",
                                          collection_name="sec_filings")


print("Indexed", len(documents), "chunks.")

Indexed 1093 chunks.


In [16]:
# --- YOU WRITE: a metadata-FILTERED search ------------------------------
# Restrict the same query to a single company's filing using a Qdrant Filter.
#   - build qmodels.Filter(must=[qmodels.FieldCondition(
#         key="metadata.ticker", match=qmodels.MatchValue(value="AAPL"))])
#   - pass it to qdrant.similarity_search(query, k=3, filter=...)
query = "What are the main risk factors facing the business?"

apple_only = qmodels.Filter(must=qmodels.FieldCondition(
    key='metadata.ticker', match=qmodels.MatchValue(value='AAPL')))
hits = qdrant.similarity_search(query, k=3, filter=apple_only)
for h in hits:
    print(h.metadata["ticker"], "|", h.page_content[:120].replace("\n", " "), "...")

AAPL | a timely basis or at all, a counterparty’s failure to perform or deliver as anticipated, or the imposition of onerous co ...
AAPL | present risks not originally contemplated, and materially adversely affect the Company’s business, reputation, results o ...
AAPL | quality and warranty costs effectively; shifts in the mix of products and services, or in the geographic, currency or ch ...


In [11]:
# --- INSTRUMENT: filter actually scopes the results ---------------------
unfiltered = qdrant.similarity_search(query, k=5)
tickers_unfiltered = [d.metadata["ticker"] for d in unfiltered]
tickers_filtered   = [d.metadata["ticker"] for d in hits]
print("Unfiltered top-5 tickers:", tickers_unfiltered)
print("AAPL-filtered  tickers:  ", tickers_filtered)
assert set(tickers_filtered) == {"AAPL"}, "filter should return only AAPL chunks"
print("Metadata filter works ✓")

Unfiltered top-5 tickers: ['NVDA', 'AAPL', 'AAPL', 'NVDA', 'NVDA']
AAPL-filtered  tickers:   ['AAPL', 'AAPL', 'AAPL']
Metadata filter works ✓


**🎤 Talk-track — Vector store.** *"A vector DB buys me two things over a raw matrix:
an approximate-nearest-neighbour index so search stays fast as the corpus grows, and metadata
filtering. That filter is doubling as a security boundary — I can constrain a query to the
documents a given user or business line is allowed to see before the LLM ever sees a token."*

---
## 04 · Retrieval — top-k, rerank

**Why.** Retrieval decides what context the LLM gets — get this wrong and no prompt can save
you. Two levers: **top-k** (how many chunks; too few misses the answer, too many buries it and
wastes the context window) and **reranking** (the fast bi-encoder search is recall-oriented;
a slower **cross-encoder** re-scores each candidate against the query for precision). Pattern:
retrieve a wide net, rerank, keep the best few.

In [12]:
# --- YOU WRITE: a top-k retriever ---------------------------------------
# qdrant.as_retriever(search_kwargs={"k": 4})
retriever =qdrant.as_retriever(search_kwargs={"k": 20})

probe = "How does the company describe competition in its industry?"
for d in retriever.invoke(probe):
    print(d.metadata["ticker"], "|", d.page_content[:100].replace("\n", " "), "...")

AAPL | the Company faces significant competition as competitors imitate the Company’s product features and  ...
AAPL | marketing, distribution and other resources, as well as established hardware, software, and service  ...
AAPL | In contrast, many of the Company’s competitors seek to compete primarily through aggressive pricing  ...
AAPL | gross margins, continual improvement in product performance, and price sensitivity on the part of co ...
NVDA | services and technologies, including those mentioned above in this Annual Report on Form 10-K, may b ...
AAPL | many critical components, a business interruption affecting such sources would exacerbate any negati ...
AAPL | and price sensitivity on the part of consumers and businesses. These markets are further defined by  ...
AAPL | to compete successfully also depends on the effective protection and enforcement of its intellectual ...
AAPL | obtaining sufficient quantities from its suppliers or in a timely manner, or in identifying and o

In [ ]:
# --- PROVIDED: load a cross-encoder reranker ----------------------------
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Reranker loaded.")

In [22]:
# --- YOU WRITE: retrieve wide, then rerank to the best few --------------
# 1) similarity_search(query, k=fetch_k) to get a wide candidate set
# 2) reranker.predict([(query, chunk_text), ...]) to score each candidate
# 3) sort candidates by score (desc) and return the top_k
def retrieve_reranked(query, fetch_k=10, top_k=4):
    ### YOUR CODE HERE
    candidates = retriever.invoke(query, k=fetch_k)

    for candidate in candidates:
      chunk_text = candidate.page_content
      score = reranker.predict([(query, chunk_text)])[0]
      candidate.metadata['score'] = score

    candidates.sort(key=lambda x: x.metadata['score'], reverse=True)
    return candidates[:top_k]

reranked = retrieve_reranked(probe)
print("Top result AFTER rerank:")
print(textwrap.fill(reranked[0].page_content[:300], 100))

Top result AFTER rerank:
the Company faces significant competition as competitors imitate the Company’s product features and
applications within their products to offer more competitive solutions. The Company also expects
competition to intensify as competitors imitate the Company’s approach to providing components
seamless


In [23]:
# --- INSTRUMENT: did reranking reorder anything? ------------------------
base = qdrant.similarity_search(probe, k=4)
def sig(docs):  # short signature of each chunk for eyeballing order
    return [d.page_content[:45].replace("\n", " ") for d in docs]
print("Bi-encoder order (top-4):")
for s in sig(base): print("  ", s)
print("\nCross-encoder rerank order (top-4):")
for s in sig(reranked): print("  ", s)

Bi-encoder order (top-4):
   the Company faces significant competition as 
   marketing, distribution and other resources, 
   In contrast, many of the Company’s competitor
   gross margins, continual improvement in produ

Cross-encoder rerank order (top-4):
   the Company faces significant competition as 
   marketing, distribution and other resources, 
   gross margins, continual improvement in produ
   In contrast, many of the Company’s competitor


**🎤 Talk-track — Retrieval.** *"Vector search optimises recall — it's a cheap bi-encoder,
so it's fast but blunt. I over-fetch, say top-10, then rerank with a cross-encoder that actually
reads the query and chunk together and scores relevance, and keep the best 4. It's the single
highest-leverage quality knob in RAG, and it's where I'd point an eval harness first."*

✍️ **Reflect:** when is reranking *not* worth its latency? (Hint: think about how well-separated
your corpus is, and your p95 latency budget.)

---
## 05 · LLM — prompt + context

**Why.** The last stage stitches retrieved context into a prompt and generates the answer. The
prompt does real work: it must **ground** the model in the context and tell it what to do when
the answer isn't there. This is also where hallucination lives — which is exactly what the
Phase-2 evaluation layer will measure.

In [ ]:
# --- PROVIDED: load Mistral-7B-Instruct in 4-bit (fits a T4) ------------
# Run this cell ONCE. Re-running stacks model copies and can OOM the GPU.
model_id = "mistralai/Mistral-7B-Instruct-v0.3"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {model_id} ...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, quantization_config=quantization_config, device_map="auto"
)

mistral_pipe = pipeline(
    "text-generation", model=model, tokenizer=tokenizer,
    max_new_tokens=512, truncation=True, temperature=0.3, top_p=0.95,
    do_sample=True, repetition_penalty=1.2, return_full_text=False,
)
mistral_pipe.model.config.pad_token_id = mistral_pipe.model.config.eos_token_id
llm = HuggingFacePipeline(pipeline=mistral_pipe)
print("LLM ready.")

In [28]:
# --- YOU WRITE: the RAG prompt + the full chain -------------------------
# 1) format_docs(docs): join retrieved chunks into one context string
#    (prefix each with its [ticker form_type] so the model can cite it)
# 2) a prompt that says: answer ONLY from context; if it's not there, say so; cite the ticker
# 3) wire the LCEL chain:
#      {"context": retriever | format_docs, "question": RunnablePassthrough()}
#      | rag_prompt | llm | StrOutputParser()
def format_docs(docs):
    ### YOUR CODE HERE
    return "\n".join([f"{d.metadata['ticker']} {d.metadata['form_type']}: {d.page_content}" for d in docs])

rag_template = """Answer the question ONLY from the provided context. If there is no context provided, say so. Do not make assumptions. Cite the ticker and the form type.

Context:
{context}

Question: {question}

Answer:"""
rag_prompt = ChatPromptTemplate.from_template(rag_template)

rag_chain = {"context": retriever | format_docs, "question": RunnablePassthrough()} | rag_prompt | llm | StrOutputParser()

print("RAG chain built.")

RAG chain built.


In [30]:
# --- RUN IT: ask the pipeline real questions ----------------------------
questions = [
    "What are the main risk factors the company highlights?",
    "How does the company describe competition in its industry?",
    "What does the company say about its revenue growth?",
]
for q in questions:
    print("Q:", q)
    print("A:", rag_chain.invoke(q).strip(), "\n" + "-"*90)

[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What are the main risk factors the company highlights?


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: From the provided context, both NVDA and AAPL highlight several risk factors that could harm their business, financial condition, results of operations, or reputation. Here's a summary of the major risk categories mentioned:

1. Industry and Market Risks: Failure to meet evolving customer needs, competition, and difficulties in estimating customer demand accurately leading to supply-demand mismatch.
2. Demand, Supply, and Manufacturing Risks: Long manufacturing lead times, uncertainty in supply and capacity availability, dependency on third-party suppliers, and potential impacts from industrial accidents.
3. Strategic and Competitive Risks: Intense competition, barriers to entry, fast technological evolution, and need for continuous innovation.
4. Financial Risks: Volatility in net sales and gross margins due to various factors, including shifts in product mix, foreign exchange rates, inflation, macroeconomic pressures, and new product introductions.
5. Reputation and Brand Risks: D

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: The company describes competition in its industry as being intense and highly competitive, characterized by aggressive price competition, downward pressure on gross margins, continual improvement in product performance, and price sensitivity on the part of consumers and businesses. It mentions that it competes with companies that have significant technical, marketing, distribution and other resources, as well as established hardware, software, and service offerings. Some competitors seek to compete primarily through aggressive pricing and very low cost structures, while others attempt to imitate the company's products and infringe on its intellectual property. The markets are further defined by frequent introduction of new products and services, short product life cycles, evolving industry standards, and rapid adoption of technological advancements by competitors. 
------------------------------------------------------------------------------------------
Q: What does the company say

OutOfMemoryError: CUDA out of memory. Tried to allocate 6.69 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.97 GiB is free. Including non-PyTorch memory, this process has 12.59 GiB memory in use. Of the allocated memory 5.61 GiB is allocated by PyTorch, and 6.84 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [31]:
# --- INSTRUMENT: inspect grounding + probe a hallucination case ---------
# (a) See exactly which chunks grounded an answer.
q = questions[0]
print("Retrieved context for:", q, "\n")
for d in retriever.invoke(q):
    print(" -", d.metadata["ticker"], "|", d.page_content[:90].replace("\n", " "), "...")

# (b) Ask something the filings can't answer — does the prompt make it decline?
print("\nOut-of-scope question:")
print(rag_chain.invoke("What is the CEO's home address?").strip())

[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Retrieved context for: What are the main risk factors the company highlights? 

 - NVDA | which could cause our stock price to decline. Additional risks, trends and uncertainties n ...
 - AAPL | the Company to manufacture and deliver products to its customers, create delays and ineffi ...
 - NVDA | our business. • We may not be able to realize the potential benefits of business investmen ...
 - AAPL | a timely basis or at all, a counterparty’s failure to perform or deliver as anticipated, o ...
 - AAPL | quality and warranty costs effectively; shifts in the mix of products and services, or in  ...
 - NVDA | from making or selling our products. • We are subject to stringent and changing data priva ...
 - AAPL | be subject to litigation or government investigations, can be liable for associated invest ...
 - AAPL | present risks not originally contemplated, and materially adversely affect the Company’s b ...
 - NVDA | test, or package our products reduces our control over product quantit

OutOfMemoryError: CUDA out of memory. Tried to allocate 4.61 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.97 GiB is free. Including non-PyTorch memory, this process has 12.59 GiB memory in use. Of the allocated memory 10.13 GiB is allocated by PyTorch, and 2.32 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

**🎤 Talk-track — LLM.** *"The prompt is a contract: answer only from the retrieved
context, decline when it's not there, and cite the source. I keep temperature low for a factual
task. The out-of-scope probe is the important one — 'answer only from context' plus an explicit
'say you don't know' is a first, cheap guardrail against hallucination. Proving how well it
actually holds is the job of the evaluation layer, which is where I'd go next."*

---
## Recap — the whole pipeline, one line per stage

| Stage | Decision you made | The tradeoff to discuss |
|-------|-------------------|-------------------------|
| **01 Ingestion** | clean HTML; recursive chunks ~1k/150 + metadata | precision vs. severed context; clean input = clean vector |
| **02 Embedding** | `all-mpnet-base-v2`, local | quality vs. cost/latency; query & corpus must share the model |
| **03 Vector store** | Qdrant + `ticker` metadata filter | ANN index scales; filter doubles as an access boundary |
| **04 Retrieval** | top-k over-fetch → cross-encoder rerank | recall (bi-encoder) vs. precision (cross-encoder) vs. latency |
| **05 LLM** | ground-only prompt + decline + cite | hallucination control; low temp for factual answers |

You can now walk any of these left-to-right on a whiteboard and defend each knob.

## Phase 2 — the evaluation / experimentation layer *(next exercise)*

This notebook stops at a *working* pipeline. The next one makes it a *measured* one — the part
of the diagram most candidates skip, and your differentiator:

1. **Gold Q/A set** — a handful of questions with known-correct answers + the source chunk.
2. **Retrieval metrics** — recall@k / MRR: did the right chunk make it into context?
3. **Answer metrics** — **faithfulness** (is every claim supported by the context?) and
   **relevance**, scored by an **LLM-as-judge (Claude)** — stronger and more credible than a
   local 7B grading itself.
4. **One experiment** — change *one* knob (e.g. rerank on/off, or chunk size), hold everything
   else fixed, and measure the effect on the same gold set instead of eyeballing it.

*That layer is where the Claude judge and any causal/experimental rigor come in — kept separate
so this notebook stays a clean, one-to-two-day full-pipeline build.*